# StudyAI PDF Pipeline (Google Colab)

This notebook shows the same idea as the website, in a simple order:

Upload PDF → Extract text → OCR if needed → Generate summary → Generate quiz → Display results

**Do not put your API key in this file.** In Colab go to the key icon (Secrets) and add `AI_API_KEY`.

In [ ]:
!pip install -q pymupdf pillow pytesseract groq
!apt-get -qq update && apt-get -qq install -y tesseract-ocr

In [ ]:
from google.colab import userdata, files
import fitz
from PIL import Image
import pytesseract
import json
from groq import Groq

AI_API_KEY = userdata.get("AI_API_KEY")
AI_MODEL = "llama-3.1-8b-instant"
MIN_TEXT_LENGTH = 80
MAX_AI_CHARS = 12000

print("Key loaded:" , bool(AI_API_KEY))

## 1. Upload a PDF

In [ ]:
uploaded = files.upload()
pdf_path = next(iter(uploaded.keys()))
print("Uploaded:", pdf_path)

## 2. Extract text (then OCR if the PDF is scanned)

In [ ]:
def extract_text_from_pdf(file_path):
    parts = []
    doc = fitz.open(file_path)
    for page in doc:
        parts.append(page.get_text() or "")
    doc.close()
    return "\n".join(parts).strip()

def ocr_pdf(file_path):
    parts = []
    doc = fitz.open(file_path)
    for page in doc:
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        parts.append(pytesseract.image_to_string(image) or "")
    doc.close()
    return "\n".join(parts).strip()

text = extract_text_from_pdf(pdf_path)
if len(text) < MIN_TEXT_LENGTH:
    print("Little text found. Trying OCR...")
    text = ocr_pdf(pdf_path)

print("Characters extracted:", len(text))
print(text[:500])

## 3. Generate summary

Flow: PDF → Extract Text → Send Text to AI → AI Summary

In [ ]:
client = Groq(api_key=AI_API_KEY)
short_text = text[:MAX_AI_CHARS]

summary_prompt = f"""Return ONLY JSON with keys overview, main_points, important_concepts, important_terms, conclusion.
Study text:\n{short_text}"""

summary_raw = client.chat.completions.create(
    model=AI_MODEL,
    messages=[{"role": "user", "content": summary_prompt}],
).choices[0].message.content

print(summary_raw)

## 4. Generate quiz

Flow: PDF → Extract Text → Send Text to AI → AI creates questions

In [ ]:
quiz_prompt = f"""Return ONLY a JSON array of 5 multiple-choice questions.
Each item: question, options, correct_answer, explanation.
Study text:\n{short_text}"""

quiz_raw = client.chat.completions.create(
    model=AI_MODEL,
    messages=[{"role": "user", "content": quiz_prompt}],
).choices[0].message.content

print(quiz_raw)

## 5. Display results

You can copy this JSON into your notes. The full website stores it in SQLite inside `app.py`.